In [1]:
import pandas as pd
import numpy as np

# 1. 加载数据
try:
    # 队友清理好的数据路径
    df = pd.read_csv('data/sales_data_clean.csv')
    print("✅ 数据成功加载！")
    
    # 2. 计算需求波动率 (CV)
    # 按品类和日期计算总销量
    daily_sales = df.groupby(['Product_Category', 'Date'])['Quantity'].sum().reset_index()
    stats = daily_sales.groupby('Product_Category')['Quantity'].agg(['mean', 'std']).reset_index()
    stats['CV'] = stats['std'] / stats['mean']
    
    # 划分风险等级 (根据 CV 值)
    def classify_risk(cv):
        if cv < 0.5: return '🟢 低风险 (稳定)'
        if cv < 1.0: return '🟡 中风险 (波动)'
        return '🔴 高风险 (不稳定)'
    
    stats['Risk_Level'] = stats['CV'].apply(classify_risk)
    
    print("\n--- 任务 2.1.1: 需求波动率分析 ---")
    display(stats[['Product_Category', 'CV', 'Risk_Level']].sort_values(by='CV', ascending=False))

    # 3. 核心品类的城市分布 (为库存分配提供依据)
    city_dist = df.groupby(['Product_Category', 'City'])['Quantity'].sum().unstack().fillna(0)
    print("\n--- 任务 2.1.2: 各品类在城市的销量分布 (Top 5 城市展示) ---")
    display(city_dist.head())

except Exception as e:
    print(f"❌ 运行失败，原因: {e}")

✅ 数据成功加载！

--- 任务 2.1.1: 需求波动率分析 ---


,Product_Category,CV,Risk_Level
7,Toys,0.754386,🟡 中风险 (波动)
1,Books,0.745318,🟡 中风险 (波动)
5,Home & Garden,0.740623,🟡 中风险 (波动)
4,Food,0.725506,🟡 中风险 (波动)
2,Electronics,0.720999,🟡 中风险 (波动)
6,Sports,0.718458,🟡 中风险 (波动)
3,Fashion,0.690414,🟡 中风险 (波动)
0,Beauty,0.669139,🟡 中风险 (波动)



--- 任务 2.1.2: 各品类在城市的销量分布 (Top 5 城市展示) ---


City,Adana,Ankara,Antalya,Bursa,Eskisehir,Gaziantep,Istanbul,Izmir,Kayseri,Konya
Product_Category,,,,,,,,,,
Beauty,113,190,111,161,63,70,409,163,74,70
Books,112,163,104,140,63,105,358,128,53,65
Electronics,113,198,115,116,48,94,354,161,91,89
Fashion,86,213,76,173,54,119,396,163,51,80
Food,53,227,99,143,65,117,328,167,67,89


In [2]:
# Task 2.1.1: Demand Volatility Analysis (English Version)
def classify_risk_en(cv):
    if cv < 0.5: return 'Low Risk (Stable)'
    if cv < 1.0: return 'Medium Risk (Volatile)'
    return 'High Risk (Unstable)'

stats['Risk_Level_EN'] = stats['CV'].apply(classify_risk_en)

print("\n--- Task 2.1.1: Product Category Volatility Analysis (CV) ---")
# Rename columns for the teacher to understand
stats_output = stats[['Product_Category', 'CV', 'Risk_Level_EN']].copy()
stats_output.columns = ['Category', 'Coefficient of Variation (CV)', 'Risk Status']
display(stats_output.sort_values(by='Coefficient of Variation (CV)', ascending=False))


--- Task 2.1.1: Product Category Volatility Analysis (CV) ---


,Category,Coefficient of Variation (CV),Risk Status
7,Toys,0.754386,Medium Risk (Volatile)
1,Books,0.745318,Medium Risk (Volatile)
5,Home & Garden,0.740623,Medium Risk (Volatile)
4,Food,0.725506,Medium Risk (Volatile)
2,Electronics,0.720999,Medium Risk (Volatile)
6,Sports,0.718458,Medium Risk (Volatile)
3,Fashion,0.690414,Medium Risk (Volatile)
0,Beauty,0.669139,Medium Risk (Volatile)


In [3]:
# Task 2.1.2: Geographic Distribution of Demand (English Version)
# This helps decide where to allocate more inventory.

print("\n--- Task 2.1.2: Sales Distribution by Category and City ---")

# Grouping by Category and City
city_dist_en = df.groupby(['Product_Category', 'City'])['Quantity'].sum().unstack().fillna(0)

# Highlighting the top cities for the teacher
display(city_dist_en)

# Professional Note for your report:
print("\n[Strategic Insight]: Istanbul consistently shows the highest demand across all categories.")
print("Inventory allocation should be prioritized for Istanbul to ensure high service levels.")


--- Task 2.1.2: Sales Distribution by Category and City ---


City,Adana,Ankara,Antalya,Bursa,Eskisehir,Gaziantep,Istanbul,Izmir,Kayseri,Konya
Product_Category,,,,,,,,,,
Beauty,113,190,111,161,63,70,409,163,74,70
Books,112,163,104,140,63,105,358,128,53,65
Electronics,113,198,115,116,48,94,354,161,91,89
Fashion,86,213,76,173,54,119,396,163,51,80
Food,53,227,99,143,65,117,328,167,67,89
Home & Garden,90,191,107,146,56,72,355,170,68,98
Sports,140,244,94,124,70,87,380,196,72,112
Toys,91,222,96,149,39,89,364,162,69,87



[Strategic Insight]: Istanbul consistently shows the highest demand across all categories.
Inventory allocation should be prioritized for Istanbul to ensure high service levels.


In [4]:
# Task 2.2: Safety Stock Level Calculation (English Version)
import numpy as np

# --- Parameters Setting ---
# Service Level: 95% (Z-score = 1.65). This means we want to avoid stockouts 95% of the time.
# Lead Time (LT): 3 days. Average time it takes to restock from the supplier.
Z_score = 1.65
Lead_Time = 3

# Safety Stock Formula: SS = Z * Standard_Deviation * sqrt(Lead_Time)
# We use 'std' calculated from Task 2.1.1
stats['Safety_Stock'] = np.ceil(Z_score * stats['std'] * np.sqrt(Lead_Time)).astype(int)

print("\n--- Task 2.2: Recommended Safety Stock Levels per Category ---")

# Creating a clean table for the report
ss_table = stats[['Product_Category', 'Safety_Stock']].sort_values(by='Safety_Stock', ascending=False)
ss_table.columns = ['Product Category', 'Suggested Safety Stock (Units)']

display(ss_table)

# Professional explanation for your teacher:
print("\n[Analysis]: The safety stock accounts for demand uncertainty during the 3-day replenishment period.")
print(f"Categories with higher volatility, like {ss_table.iloc[0,0]}, require a larger buffer of {ss_table.iloc[0,1]} units.")


--- Task 2.2: Recommended Safety Stock Levels per Category ---


,Product Category,Suggested Safety Stock (Units)
6,Sports,10
7,Toys,10
0,Beauty,9
1,Books,9
2,Electronics,9
3,Fashion,9
4,Food,9
5,Home & Garden,9



[Analysis]: The safety stock accounts for demand uncertainty during the 3-day replenishment period.
Categories with higher volatility, like Sports, require a larger buffer of 10 units.


In [5]:
# Task 2.3: Inventory Replenishment Strategy (English Version)

# Define Replenishment Strategy based on CV
def replenishment_strategy(cv):
    if cv < 0.7:  # More stable
        return 'Periodic Review (Fixed Interval)'
    else:         # More volatile
        return 'Continuous Review (Reorder Point)'

stats['Strategy'] = stats['CV'].apply(replenishment_strategy)

# Calculate Reorder Point (ROP)
# ROP = (Average Daily Demand * Lead Time) + Safety Stock
stats['Reorder_Point'] = np.ceil((stats['mean'] * Lead_Time) + stats['Safety_Stock']).astype(int)

print("\n--- Task 2.3: Final Inventory Strategy Recommendation ---")

final_report = stats[['Product_Category', 'Strategy', 'Reorder_Point', 'Safety_Stock']]
final_report.columns = ['Category', 'Replenishment Strategy', 'Reorder Point (ROP)', 'Safety Stock']

display(final_report)

# Summary for the teacher
print("\n[Executive Summary]:")
print("1. Stable categories use 'Periodic Review' to reduce management costs.")
print("2. Volatile categories use 'Continuous Review' with a calculated 'Reorder Point' to prevent stockouts.")
print("3. When Inventory Level <= Reorder Point, a new order must be placed.")


--- Task 2.3: Final Inventory Strategy Recommendation ---


,Category,Replenishment Strategy,Reorder Point (ROP),Safety Stock
0,Beauty,Periodic Review (Fixed Interval),22,9
1,Books,Continuous Review (Reorder Point),21,9
2,Electronics,Continuous Review (Reorder Point),22,9
3,Fashion,Periodic Review (Fixed Interval),22,9
4,Food,Continuous Review (Reorder Point),22,9
5,Home & Garden,Continuous Review (Reorder Point),22,9
6,Sports,Continuous Review (Reorder Point),24,10
7,Toys,Continuous Review (Reorder Point),23,10



[Executive Summary]:
1. Stable categories use 'Periodic Review' to reduce management costs.
2. Volatile categories use 'Continuous Review' with a calculated 'Reorder Point' to prevent stockouts.
3. When Inventory Level <= Reorder Point, a new order must be placed.
